# Brazilian Marketplace (Olist) - Business Analysis

Olist is a Brazilian marketplace connecting small businesses to a wide network of online sales channels. This notebook analyzes ~100,000 orders placed on the Olist Store between 2016 and 2018, using the public [Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0).

**Business question:** How can Olist optimize its operational and marketing strategy to improve profitability, increase customer satisfaction, and reduce delivery times, based on historical sales data, customer reviews, and logistics metrics?

**Method:** SQL on Databricks - data modeling, cleaning, revenue analysis, AOV, CLV, RFM segmentation, churn rate, and correlation studies.

## 1. Setup & Schema Overview

The 9 source CSVs were loaded as tables in `workspace.brazilian_marketplace`. Before analyzing, let's confirm the schema of each table matches the dataset documentation.

In [0]:
-- I'm checking the schema of every table I loaded, to make sure it matches what I expect from the dataset documentation before I do anything else with it

SELECT table_name, column_name, data_type, ordinal_position
FROM workspace.information_schema.columns
WHERE table_schema = 'brazilian_marketplace'
ORDER BY table_name, ordinal_position;

The schema matches the dataset documentation for 7 of 9 tables. Two exceptions worth noting:

- `product_category_name_translation` carries an extra `dummy_column` (a constant value on every row, not useful for anything). I'll exclude it from all queries going forward.
- `reviews` only has `review_id`, `order_id`, `review_score` - the source file for reviews is a lighter version that skips the free-text comments and survey dates. That's fine here, since this analysis only needs the review score.

Row counts per table, as a sanity check against the documented dataset size:


In [0]:
-- Quick sanity check: do my row counts match what the Olist dataset documentation says each table should have?

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM workspace.brazilian_marketplace.customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM workspace.brazilian_marketplace.geolocation
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.brazilian_marketplace.order_items
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.brazilian_marketplace.orders
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.brazilian_marketplace.payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.brazilian_marketplace.reviews
UNION ALL SELECT 'products', COUNT(*) FROM workspace.brazilian_marketplace.products
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.brazilian_marketplace.sellers
UNION ALL SELECT 'product_category_name_translation', COUNT(*) FROM workspace.brazilian_marketplace.product_category_name_translation;

All counts match the documented Olist dataset sizes (e.g. 99,441 customers and orders, 112,650 order items, 1,000,163 geolocation records) - the load is clean.

## 2. Primary Key Identification

For each table, a column is a primary key candidate if `COUNT(*) = COUNT(DISTINCT column)` - every value is present and unique.

In [0]:
-- For each table, I'm testing whether a column could be a primary key: if COUNT(*) equals COUNT(DISTINCT column), every value is unique and present, so it's a valid single-column key

SELECT 'customers' AS table_name, 'customer_id' AS candidate_column,
       COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_non_null,
       COUNT(*) = COUNT(DISTINCT customer_id) AS is_primary_key
FROM workspace.brazilian_marketplace.customers
UNION ALL
SELECT 'orders', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.orders
UNION ALL
SELECT 'products', 'product_id', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) = COUNT(DISTINCT product_id)
FROM workspace.brazilian_marketplace.products
UNION ALL
SELECT 'sellers', 'seller_id', COUNT(*), COUNT(DISTINCT seller_id), COUNT(*) = COUNT(DISTINCT seller_id)
FROM workspace.brazilian_marketplace.sellers
UNION ALL
SELECT 'reviews', 'review_id', COUNT(*), COUNT(DISTINCT review_id), COUNT(*) = COUNT(DISTINCT review_id)
FROM workspace.brazilian_marketplace.reviews
UNION ALL
SELECT 'product_category_name_translation', 'product_category_name', COUNT(*), COUNT(DISTINCT product_category_name), COUNT(*) = COUNT(DISTINCT product_category_name)
FROM workspace.brazilian_marketplace.product_category_name_translation
UNION ALL
SELECT 'order_items', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'geolocation', 'geolocation_zip_code_prefix', COUNT(*), COUNT(DISTINCT geolocation_zip_code_prefix), COUNT(*) = COUNT(DISTINCT geolocation_zip_code_prefix)
FROM workspace.brazilian_marketplace.geolocation;

Four tables came back `false` above. For `order_items`, `payments`, and `geolocation` this is expected (an order can have several items or payment installments; a zip prefix covers many lat/lng points). But `reviews.review_id` repeating is unexpected - a review should be unique. Let's look closer before settling on a composite key.


In [0]:
-- I want to see which review_ids repeat and how often, to understand why review_id alone isn't unique
SELECT review_id, COUNT(*) AS n_occurrences
FROM workspace.brazilian_marketplace.reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY n_occurrences DESC
LIMIT 10;

In [0]:
-- Let's look at the actual rows behind one of these duplicated review_ids, to see what's different between the copies
SELECT *
FROM workspace.brazilian_marketplace.reviews
WHERE review_id IN (
  SELECT review_id
  FROM workspace.brazilian_marketplace.reviews
  GROUP BY review_id
  HAVING COUNT(*) > 1
)
ORDER BY review_id
LIMIT 20;

The duplicates differ by `order_id`: Olist occasionally ties one review survey to more than one order (e.g. orders placed close together get bundled into a single satisfaction survey). So `review_id` alone isn't a key, but `review_id` + `order_id` together should be. Let's confirm that, along with the other composite keys.


In [0]:
-- Checking composite keys for the three tables that failed the single-column test: order_id + order_item_id for order_items, order_id + payment_sequential for payments, review_id + order_id for reviews
SELECT 'order_items' AS table_name, COUNT(*) AS total_rows,
       COUNT(DISTINCT CONCAT(order_id, '-', order_item_id)) AS distinct_composite
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', COUNT(*), COUNT(DISTINCT CONCAT(order_id, '-', payment_sequential))
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT CONCAT(review_id, '-', order_id))
FROM workspace.brazilian_marketplace.reviews;


## Primary Key Summary

| Table | Primary key |
|---|---|
| customers | `customer_id` |
| orders | `order_id` |
| products | `product_id` |
| sellers | `seller_id` |
| product_category_name_translation | `product_category_name` |
| order_items | `order_id` + `order_item_id` (composite) |
| payments | `order_id` + `payment_sequential` (composite) |
| reviews | `review_id` + `order_id` (composite) |
| geolocation | no single natural key - it's a reference lookup table (many lat/lng points share a zip prefix); not decomposed further, since it's only ever joined on `geolocation_zip_code_prefix` |

## 3. ER Schema / Data Model

Relationships between the tables:

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : places
    ORDERS ||--o{ ORDER_ITEMS : contains
    ORDERS ||--o{ PAYMENTS : "paid via"
    ORDERS ||--o{ REVIEWS : receives
    ORDER_ITEMS }o--|| PRODUCTS : references
    ORDER_ITEMS }o--|| SELLERS : "sold by"
    PRODUCTS }o--|| PRODUCT_CATEGORY_NAME_TRANSLATION : "category in"
    CUSTOMERS }o--|| GEOLOCATION : "zip code in"
    SELLERS }o--|| GEOLOCATION : "zip code in"
```

## 4. Data Cleaning - Missing Values

I'll check every table for NULLs across all of its columns in one pass, then decide table by table whether a NULL can actually be fixed or has to stay as-is.

In [0]:
-- I'm checking, for every table, how many rows have at least one NULL in any column - a quick way to see where missing data actually is before deciding what to do about it
SELECT 'customers' AS table_name, COUNT(*) AS rows_with_nulls
FROM workspace.brazilian_marketplace.customers
WHERE customer_id IS NULL OR customer_unique_id IS NULL OR customer_zip_code_prefix IS NULL OR customer_city IS NULL OR customer_state IS NULL
UNION ALL
SELECT 'geolocation', COUNT(*)
FROM workspace.brazilian_marketplace.geolocation
WHERE geolocation_zip_code_prefix IS NULL OR geolocation_lat IS NULL OR geolocation_lng IS NULL OR geolocation_city IS NULL OR geolocation_state IS NULL
UNION ALL
SELECT 'order_items', COUNT(*)
FROM workspace.brazilian_marketplace.order_items
WHERE order_id IS NULL OR order_item_id IS NULL OR product_id IS NULL OR seller_id IS NULL OR shipping_limit_date IS NULL OR price IS NULL OR freight_value IS NULL
UNION ALL
SELECT 'orders', COUNT(*)
FROM workspace.brazilian_marketplace.orders
WHERE order_id IS NULL OR customer_id IS NULL OR order_status IS NULL OR order_purchase_timestamp IS NULL OR order_approved_at IS NULL OR order_delivered_carrier_date IS NULL OR order_delivered_customer_date IS NULL OR order_estimated_delivery_date IS NULL
UNION ALL
SELECT 'payments', COUNT(*)
FROM workspace.brazilian_marketplace.payments
WHERE order_id IS NULL OR payment_sequential IS NULL OR payment_type IS NULL OR payment_installments IS NULL OR payment_value IS NULL
UNION ALL
SELECT 'reviews', COUNT(*)
FROM workspace.brazilian_marketplace.reviews
WHERE review_id IS NULL OR order_id IS NULL OR review_score IS NULL
UNION ALL
SELECT 'products', COUNT(*)
FROM workspace.brazilian_marketplace.products
WHERE product_id IS NULL OR product_category_name IS NULL OR product_name_lenght IS NULL OR product_description_lenght IS NULL OR product_photos_qty IS NULL OR product_weight_g IS NULL OR product_length_cm IS NULL OR product_height_cm IS NULL OR product_width_cm IS NULL
UNION ALL
SELECT 'sellers', COUNT(*)
FROM workspace.brazilian_marketplace.sellers
WHERE seller_id IS NULL OR seller_zip_code_prefix IS NULL OR seller_city IS NULL OR seller_state IS NULL
UNION ALL
SELECT 'product_category_name_translation', COUNT(*)
FROM workspace.brazilian_marketplace.product_category_name_translation
WHERE product_category_name IS NULL OR product_category_name_english IS NULL;

`orders` and `products` are the tables most likely to show NULLs here: some orders were never approved or delivered, so their date columns are legitimately empty, and a small number of products are missing their category name and dimensions in the source data. Without more information from Olist, most of these can't be fixed and are left as-is - that's a real, documented limitation of the dataset, not something to paper over.

The one gap worth closing is the missing **English category translations** in `products`.

Products only carry the Portuguese category name. I'm adding an English version by joining in the translation table - and since a couple of categories don't have a translation on record, I'm filling those in by hand, with anything still unmatched labeled `'N/A'` so it doesn't silently disappear from later analysis.


In [0]:
-- Rebuilding products with an English category column: joined from the translation table, with the two known missing translations filled in by hand, and 'N/A' for anything left unmatched. Safe to run this as many times as I want, from any starting point.
CREATE OR REPLACE TABLE workspace.brazilian_marketplace.products AS
SELECT
  p.product_id,
  p.product_category_name,
  p.product_name_lenght,
  p.product_description_lenght,
  p.product_photos_qty,
  p.product_weight_g,
  p.product_length_cm,
  p.product_height_cm,
  p.product_width_cm,
  COALESCE(
    t.product_category_name_english,
    CASE
      WHEN p.product_category_name = 'portateis_cozinha_e_preparadores_de_alimentos' THEN 'kitchen_and_food_preparation_portable_devices'
      WHEN p.product_category_name = 'pc_gamer' THEN 'gaming_pc'
      ELSE 'N/A'
    END
  ) AS product_category_name_eng
FROM workspace.brazilian_marketplace.products AS p
LEFT JOIN workspace.brazilian_marketplace.product_category_name_translation AS t
  ON p.product_category_name = t.product_category_name;

In [0]:
-- Quick check that every product now has an English category: this should return 0
SELECT COUNT(*) AS still_missing
FROM workspace.brazilian_marketplace.products
WHERE product_category_name_eng IS NULL;

## 5. Data Cleaning - Duplicates

I already answered this back in Step 2: every table with a natural identifier (`customer_id`, `order_id`, `product_id`, `seller_id`, `review_id` + `order_id`, etc.) came back unique when tested as a primary key candidate. So there's no separate duplicate-row problem here - the primary key check *is* the duplicate check for this dataset.

## 6. Revenue Analysis - Total Revenue, Valid Order Statuses, Time Span

Before calculating revenue, I need to know which order statuses actually represent completed sales, and how much history the data covers.

In [0]:
-- Let's see what order statuses exist, so I know which ones should count toward revenue
SELECT DISTINCT order_status
FROM workspace.brazilian_marketplace.orders
ORDER BY order_status;

Olist's own documentation treats only `"delivered"` orders as completed sales - everything else (created, shipped, canceled, etc.) hasn't actually generated revenue yet, or never will. I'll filter on that status for every revenue calculation from here on.


In [0]:
-- Total revenue: summing payment_value for delivered orders only
SELECT ROUND(SUM(p.payment_value), 0) AS total_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';

That revenue figure only means something with a time frame attached. Let's find the first and last purchase dates, and the span between them in days, weeks, months, and years.


In [0]:
-- Using datediff() with a unit argument to get the gap between the first and last purchase in several different units at once
SELECT
  MIN(order_purchase_timestamp) AS started_time,
  MAX(order_purchase_timestamp) AS ended_time,
  DATEDIFF(DAY, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS days,
  DATEDIFF(WEEK, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS weeks,
  DATEDIFF(MONTH, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS months,
  DATEDIFF(YEAR, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS years
FROM workspace.brazilian_marketplace.orders;

This gives me the full picture: how much revenue Olist generated, and over what period. That's the baseline everything else in this analysis builds on.


## 7. Revenue Analysis - Periodic Averages

Now that I have the total revenue and the time span, I can break that down into average revenue per day, week, month, and year - useful for spotting whether performance in any given period is above or below the norm.

In [0]:
-- Combining total revenue with the time span to get the average revenue per day, week, month, and year
WITH total_rev AS (
  SELECT SUM(p.payment_value) AS total_revenue
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.payments AS p
    ON o.order_id = p.order_id
  WHERE o.order_status = 'delivered'
),
time_span AS (
  SELECT
    DATEDIFF(DAY, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS days,
    DATEDIFF(WEEK, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS weeks,
    DATEDIFF(MONTH, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS months,
    DATEDIFF(YEAR, MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) AS years
  FROM workspace.brazilian_marketplace.orders
)
SELECT
  ROUND(total_rev.total_revenue / time_span.days, 0) AS avg_revenue_per_day,
  ROUND(total_rev.total_revenue / time_span.weeks, 0) AS avg_revenue_per_week,
  ROUND(total_rev.total_revenue / time_span.months, 0) AS avg_revenue_per_month,
  ROUND(total_rev.total_revenue / time_span.years, 0) AS avg_revenue_per_year
FROM total_rev
CROSS JOIN time_span;

These averages give me a quick benchmark: I can compare any specific day, week, month, or year against them to spot whether Olist is over- or under-performing in that period.


## 8. Revenue Trends - Annual and Quarterly

Let's look at how revenue actually moved over time, first by year and then broken down by quarter, to see the shape of that growth.

In [0]:
-- Revenue by year, delivered orders only
SELECT
  YEAR(o.order_purchase_timestamp) AS year,
  ROUND(SUM(p.payment_value), 0) AS annual_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY YEAR(o.order_purchase_timestamp)
ORDER BY year;

Now the same thing broken down by quarter, to see the trend in more detail.


In [0]:
-- Revenue by year and quarter, delivered orders only
SELECT
  YEAR(o.order_purchase_timestamp) AS year,
  QUARTER(o.order_purchase_timestamp) AS quarter,
  ROUND(SUM(p.payment_value), 0) AS quarterly_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY YEAR(o.order_purchase_timestamp), QUARTER(o.order_purchase_timestamp)
ORDER BY year, quarter;

Since the data only starts in September 2016, that first year will be a partial one - worth keeping in mind when comparing it to 2017 and 2018.


## 9. Seasonality of Sales

Let's see whether some months consistently do better than others, and check if the growth trend also shows any seasonal pattern within the year.

In [0]:
-- Revenue by year, quarter, and month, delivered orders only
SELECT
  YEAR(o.order_purchase_timestamp) AS year,
  QUARTER(o.order_purchase_timestamp) AS quarter,
  MONTH(o.order_purchase_timestamp) AS month,
  ROUND(SUM(p.payment_value), 0) AS monthly_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY YEAR(o.order_purchase_timestamp), QUARTER(o.order_purchase_timestamp), MONTH(o.order_purchase_timestamp)
ORDER BY year, quarter, month;

2016 and 2018 are both partial years in this data (it starts in September 2016 and ends in October 2018), so a fair month-by-month comparison across years isn't really possible for those two. I'll look at average revenue per calendar month across the whole dataset instead, which smooths that out.


In [0]:
-- Average revenue per calendar month, across the whole dataset
SELECT
  MONTH(o.order_purchase_timestamp) AS month,
  ROUND(AVG(p.payment_value), 0) AS avg_monthly_revenue
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY MONTH(o.order_purchase_timestamp)
ORDER BY month;

This shows whether certain months consistently bring in higher average revenue per order than others, independent of which year they fall in.


## 10. Top Popular Products and Categories

Let's see which individual products sell the most, and which categories drive the most volume and revenue overall.

In [0]:
-- Top 10 products by number of items sold, delivered orders only
SELECT
  oi.product_id,
  COUNT(*) AS items_sold,
  p.product_category_name_eng
FROM workspace.brazilian_marketplace.orders AS o
INNER JOIN workspace.brazilian_marketplace.order_items AS oi
  ON o.order_id = oi.order_id
INNER JOIN workspace.brazilian_marketplace.products AS p
  ON oi.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY oi.product_id, p.product_category_name_eng
ORDER BY items_sold DESC
LIMIT 10;

Individual product IDs aren't very meaningful on their own, so let's zoom out to categories: which ones sell the most items, and which generate the most revenue.


In [0]:
-- Top 5 product categories by items sold, alongside their share of total items and total revenue
WITH category_summary AS (
  SELECT
    pr.product_category_name_eng,
    COUNT(*) AS items_sold,
    ROUND(SUM(oi.price), 0) AS revenue
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.order_items AS oi
    ON o.order_id = oi.order_id
  INNER JOIN workspace.brazilian_marketplace.products AS pr
    ON oi.product_id = pr.product_id
  WHERE o.order_status = 'delivered'
  GROUP BY pr.product_category_name_eng
),
totals AS (
  SELECT SUM(items_sold) AS total_items, SUM(revenue) AS total_revenue
  FROM category_summary
)
SELECT
  cs.product_category_name_eng,
  cs.items_sold,
  cs.revenue,
  ROUND(cs.items_sold * 100.0 / t.total_items, 2) AS percent_of_items_sold,
  ROUND(cs.revenue * 100.0 / t.total_revenue, 2) AS percent_of_revenue
FROM category_summary AS cs
CROSS JOIN totals AS t
ORDER BY cs.items_sold DESC
LIMIT 5;

Comparing the two percentage columns shows whether a category's revenue share matches its share of items sold, or if it's over- or under-indexed - a category with a much higher revenue share than item share likely has higher-priced products on average.


## 11. AOV - Average Order Value

Average order value tells me how much a typical completed order is worth - a useful benchmark for evaluating pricing and promotions.

In [0]:
-- Average order value: total revenue divided by the number of delivered orders
WITH total_rev AS (
  SELECT SUM(p.payment_value) AS total_revenue
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.payments AS p
    ON o.order_id = p.order_id
  WHERE o.order_status = 'delivered'
),
total_orders AS (
  SELECT COUNT(DISTINCT order_id) AS n_orders
  FROM workspace.brazilian_marketplace.orders
  WHERE order_status = 'delivered'
)
SELECT
  ROUND(total_rev.total_revenue / total_orders.n_orders, 2) AS avg_order_value
FROM total_rev
CROSS JOIN total_orders;

This is my baseline order value - I'll come back to it when looking at customer lifetime value next, since the two are closely related.


## 12. CLV - Monthly Average Across the Customer Base

Customer Lifetime Value estimates how much revenue a customer is worth over their whole relationship with Olist. I'll build it up from three pieces: average purchase value, average purchase frequency, and average customer lifespan - then combine them into a monthly view.

The formula: **Average Customer Value = Average Purchase Value × Average Purchase Frequency Rate**, and then **CLV = Average Customer Value × Average Customer Lifespan**. I'll calculate the first two per month (since purchase behavior can shift over time), and the lifespan once across the whole customer base (it needs the full history to mean anything).


In [0]:
-- Building monthly CLV: average purchase value and frequency per month, combined with the overall average customer lifespan
WITH monthly_stats AS (
  SELECT
    DATE_TRUNC('MONTH', o.order_purchase_timestamp) AS month,
    SUM(p.payment_value) AS monthly_revenue,
    COUNT(DISTINCT o.order_id) AS monthly_purchases,
    COUNT(DISTINCT c.customer_unique_id) AS monthly_customers
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.payments AS p
    ON o.order_id = p.order_id
  INNER JOIN workspace.brazilian_marketplace.customers AS c
    ON o.customer_id = c.customer_id
  WHERE o.order_status = 'delivered'
  GROUP BY DATE_TRUNC('MONTH', o.order_purchase_timestamp)
),
customer_lifespans AS (
  SELECT
    c.customer_unique_id,
    DATEDIFF(MONTH, MIN(o.order_purchase_timestamp), MAX(o.order_purchase_timestamp)) AS lifespan_months
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.customers AS c
    ON o.customer_id = c.customer_id
  WHERE o.order_status = 'delivered'
  GROUP BY c.customer_unique_id
),
avg_lifespan AS (
  SELECT AVG(lifespan_months) AS avg_lifespan_months
  FROM customer_lifespans
)
SELECT
  ms.month,
  ROUND(ms.monthly_revenue / ms.monthly_purchases, 2) AS avg_purchase_value,
  ROUND(ms.monthly_purchases / ms.monthly_customers, 2) AS avg_purchase_frequency,
  ROUND((ms.monthly_revenue / ms.monthly_customers) * al.avg_lifespan_months, 2) AS clv
FROM monthly_stats AS ms
CROSS JOIN avg_lifespan AS al
ORDER BY ms.month;

This gives me a monthly CLV trend, using each month's own purchase behavior combined with the overall average customer lifespan. Next I'll break CLV down to the individual customer level, which is more useful for segmentation.


## 13. CLV - Per Individual Customer

Instead of a monthly average, this breaks CLV down to each customer individually - same formula, but calculated per person. This is what actually matters for segmentation: spotting who the high-value customers are.

In [0]:
-- CLV for each individual customer: average order value times number of orders times lifespan in months
WITH customer_orders AS (
  SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(AVG(p.payment_value), 2) AS avg_order_value
  FROM workspace.brazilian_marketplace.customers AS c
  INNER JOIN workspace.brazilian_marketplace.orders AS o
    ON c.customer_id = o.customer_id
  INNER JOIN workspace.brazilian_marketplace.payments AS p
    ON o.order_id = p.order_id
  WHERE o.order_status = 'delivered'
  GROUP BY c.customer_unique_id
),
customer_lifetime AS (
  SELECT
    c.customer_unique_id,
    DATEDIFF(MONTH, MIN(o.order_purchase_timestamp), MAX(o.order_purchase_timestamp)) AS lifetime_months
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.customers AS c
    ON o.customer_id = c.customer_id
  WHERE o.order_status = 'delivered'
  GROUP BY c.customer_unique_id
)
SELECT
  co.customer_unique_id,
  co.avg_order_value,
  co.total_orders,
  cl.lifetime_months,
  ROUND(co.avg_order_value * co.total_orders * cl.lifetime_months, 2) AS clv
FROM customer_orders AS co
INNER JOIN customer_lifetime AS cl
  ON co.customer_unique_id = cl.customer_unique_id
ORDER BY clv DESC;

Most customers here will show a CLV of 0. That's not a bug, it just means they've only ordered once, so there's no lifespan to multiply by yet. The customers with a positive CLV are the ones who came back, and they're the interesting ones going forward.


## 14. CLV - Average Across All Customers

Zooming out one more level: one single number for the whole customer base, plus the three components that make it up.

In [0]:
-- Average CLV across all customers, along with the average of each component that makes it up
WITH customer_orders AS (
  SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    AVG(p.payment_value) AS avg_order_value
  FROM workspace.brazilian_marketplace.customers AS c
  INNER JOIN workspace.brazilian_marketplace.orders AS o
    ON c.customer_id = o.customer_id
  INNER JOIN workspace.brazilian_marketplace.payments AS p
    ON o.order_id = p.order_id
  WHERE o.order_status = 'delivered'
  GROUP BY c.customer_unique_id
),
customer_lifetime AS (
  SELECT
    c.customer_unique_id,
    DATEDIFF(MONTH, MIN(o.order_purchase_timestamp), MAX(o.order_purchase_timestamp)) AS lifetime_months
  FROM workspace.brazilian_marketplace.orders AS o
  INNER JOIN workspace.brazilian_marketplace.customers AS c
    ON o.customer_id = c.customer_id
  WHERE o.order_status = 'delivered'
  GROUP BY c.customer_unique_id
),
customer_clv AS (
  SELECT
    co.customer_unique_id,
    co.avg_order_value,
    co.total_orders,
    cl.lifetime_months,
    co.avg_order_value * co.total_orders * cl.lifetime_months AS clv
  FROM customer_orders AS co
  INNER JOIN customer_lifetime AS cl
    ON co.customer_unique_id = cl.customer_unique_id
)
SELECT
  ROUND(AVG(avg_order_value), 2) AS avg_order_value_all_customers,
  ROUND(AVG(total_orders), 2) AS avg_orders_all_customers,
  ROUND(AVG(lifetime_months), 2) AS avg_lifespan_months_all_customers,
  ROUND(AVG(clv), 2) AS avg_clv_all_customers
FROM customer_clv;

This average CLV is a useful benchmark on its own: it's roughly what Olist should expect to earn from a typical customer over their relationship - useful for deciding how much is reasonable to spend acquiring one.


## 15. RFM Segmentation - Recency, Frequency, Monetary

To segment customers by value, I need three numbers for each one: how recently they last ordered, how often they order, and how much they've spent in total. I'll save this as its own table since I'll build on it in the next step.

Recency needs a reference point in time - I'm using the most recent purchase date in the whole dataset as "today", since there's no real current date to anchor this analysis to.


In [0]:
-- Building a reusable table with Recency, Frequency, and Monetary value for each customer
CREATE OR REPLACE TABLE workspace.brazilian_marketplace.rfm_customers AS
WITH report_end AS (
  SELECT MAX(order_purchase_timestamp) AS ended_time
  FROM workspace.brazilian_marketplace.orders
)
SELECT
  c.customer_unique_id,
  DATEDIFF(DAY, MAX(o.order_purchase_timestamp), (SELECT ended_time FROM report_end)) AS recency,
  COUNT(o.order_id) AS frequency,
  ROUND(SUM(p.payment_value), 2) AS monetary
FROM workspace.brazilian_marketplace.customers AS c
INNER JOIN workspace.brazilian_marketplace.orders AS o
  ON c.customer_id = o.customer_id
INNER JOIN workspace.brazilian_marketplace.payments AS p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id;

A quick look at the top spenders, just to sanity-check the numbers:


In [0]:
-- Quick look at the result, sorted by monetary value
SELECT * FROM workspace.brazilian_marketplace.rfm_customers
ORDER BY monetary DESC
LIMIT 10;

With recency, frequency, and monetary value saved for every customer, I can now split them into value-based segments - that's next.


## 16. RFM Segmentation - Customer Group Classification

Now I'll turn recency, frequency, and monetary value into scores I can compare, and use those scores to sort customers into named segments.

I'm splitting each of the three metrics into quintiles (score 1 to 5), so a score of 5 always means "best" - most recent purchase, most frequent buyer, or highest spender. I'm also breaking ties by customer ID, so the scores come out the same every time I run this.


In [0]:
-- Splitting recency, frequency, and monetary into quintile scores (5 = best on that dimension)
CREATE OR REPLACE TABLE workspace.brazilian_marketplace.rfm_customers AS
SELECT
  customer_unique_id,
  recency,
  frequency,
  monetary,
  NTILE(5) OVER (ORDER BY recency DESC, customer_unique_id) AS r_score,
  NTILE(5) OVER (ORDER BY frequency ASC, customer_unique_id) AS f_score,
  NTILE(5) OVER (ORDER BY monetary ASC, customer_unique_id) AS m_score
FROM workspace.brazilian_marketplace.rfm_customers;

With scores in hand, I can group customers into segments that are actually useful for marketing decisions:

- **High Value** - big spenders who also buy often (top monetary and frequency scores)
- **At Risk** - haven't purchased in a long time (lowest recency scores), worth a win-back push
- **Loyal** - buy often, even if not necessarily big spenders
- **Promising** - bought recently, worth nurturing into repeat customers
- **Other** - everyone who doesn't clearly fit one of the above

I check these in that order, since a customer could technically qualify for more than one - High Value and At Risk take priority since they're the most actionable for the business.

In [0]:
-- Assigning each customer to a single segment based on their RFM scores
CREATE OR REPLACE TABLE workspace.brazilian_marketplace.rfm_customers AS
SELECT
  *,
  CASE
    WHEN m_score >= 4 AND f_score >= 4 THEN 'High Value'
    WHEN r_score <= 2 THEN 'At Risk'
    WHEN f_score >= 4 THEN 'Loyal'
    WHEN r_score >= 4 THEN 'Promising'
    ELSE 'Other'
  END AS segment
FROM workspace.brazilian_marketplace.rfm_customers;

How many customers landed in each segment:


In [0]:
-- Customer count per segment
SELECT segment, COUNT(*) AS n_customers
FROM workspace.brazilian_marketplace.rfm_customers
GROUP BY segment
ORDER BY n_customers DESC;

This segmentation is the foundation for the churn analysis coming up next - especially the At Risk group, which is the clearest signal of customers who might already be gone.


## 17. Churn Rate Analysis

Before calculating a churn rate, it's worth checking something first: how many customers actually come back for a second order at all? That number changes how I should read a churn rate.

In [0]:
-- What share of customers placed more than one order?
SELECT
  COUNT(*) AS total_customers,
  SUM(CASE WHEN frequency > 1 THEN 1 ELSE 0 END) AS repeat_customers,
  ROUND(SUM(CASE WHEN frequency > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS repeat_purchase_rate_pct
FROM workspace.brazilian_marketplace.rfm_customers;

This matters because a "churn rate" usually assumes most customers are repeat buyers who eventually stop — if the vast majority only ever order once, a high churn number doesn't mean the same thing it would for a subscription business. With that context in mind, here's the churn rate using a 90-day inactivity threshold, a common baseline in e-commerce:


In [0]:
-- Churn rate: share of customers who haven't purchased in over 90 days
SELECT
  COUNT(*) AS total_customers,
  SUM(CASE WHEN recency > 90 THEN 1 ELSE 0 END) AS churned_customers,
  ROUND(SUM(CASE WHEN recency > 90 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS churn_rate_pct
FROM workspace.brazilian_marketplace.rfm_customers;

I'll read these two numbers together: the churn rate on its own would look alarming, but it's largely explained by how few customers ever return for a second purchase in the first place.
